In [7]:
import random
import time
from collections import defaultdict

# ============================================================
# 1️⃣ 시스템 상수 정의
# ============================================================

ANGLE_MIN = 0
ANGLE_MAX = 10

# 전류 = 각도 × 0.1
CURRENT_GAIN = 0.1

# 전류 보정 step
CURRENT_STEP = 0.1

# 전류 0.1 → 각도 1 변화
ANGLE_PER_CURRENT = 10.0

# 강화학습 행동
# 0: 전류 감소, 1: 유지, 2: 증가
ACTIONS = [-1, 0, 1]

# ============================================================
# 2️⃣ Q-Learning 설정
# ============================================================

Q = defaultdict(lambda: [0.0, 0.0, 0.0])

alpha = 0.1      # 학습률
gamma = 0.9      # 할인율
epsilon = 0.2    # 탐험 확률

# ============================================================
# 3️⃣ 유틸 함수
# ============================================================

def is_even_target(angle):
    """각도가 2의 배수인지 판단"""
    return angle % 2 == 0


def nearest_even(angle):
    """가장 가까운 2의 배수 각도 계산"""
    return round(angle / 2) * 2


def get_state(angle_error):
    """
    상태 정의
    - 목표 각도(2의 배수) 대비 오차
    """
    return int(angle_error)


def choose_action(state):
    """ε-greedy 정책"""
    if random.random() < epsilon:
        return random.randint(0, 2)
    return max(range(3), key=lambda a: Q[state][a])


def apply_action(current, action_idx):
    """
    전류 보정 적용
    """
    delta_i = ACTIONS[action_idx] * CURRENT_STEP
    return current + delta_i, delta_i


# ============================================================
# 4️⃣ 강화학습 사전 학습
# ============================================================

def train_agent(episodes=1000):
    """
    전류를 조정해서 각도를 2의 배수로 맞추는 학습
    """
    for _ in range(episodes):
        angle = random.randint(ANGLE_MIN, ANGLE_MAX)
        target = nearest_even(angle)
        error = angle - target
        current = angle * CURRENT_GAIN

        while error != 0:
            state = get_state(error)
            action = choose_action(state)

            current, delta_i = apply_action(current, action)

            # 전류 변화 → 각도 변화
            angle -= delta_i * ANGLE_PER_CURRENT
            next_error = angle - target

            reward = -abs(next_error)
            next_state = get_state(next_error)

            Q[state][action] += alpha * (
                reward + gamma * max(Q[next_state]) - Q[state][action]
            )

            error = next_error


train_agent()

# ============================================================
# 5️⃣ 실시간 데이터 스트림 + 제어
# ============================================================

invalid_count = 0

print("\n🚀 실시간 입력 시작\n")

while True:
    # --------------------------------------------------------
    # 입력 데이터 생성
    # --------------------------------------------------------
    angle = random.randint(ANGLE_MIN, ANGLE_MAX)
    current = angle * CURRENT_GAIN

    print(f"입력 각도: {angle}°, 전류: {current:.2f}A")

    # --------------------------------------------------------
    # 기준 위반 체크
    # --------------------------------------------------------
    if not is_even_target(angle):
        invalid_count += 1
    else:
        invalid_count = 0

    # --------------------------------------------------------
    # 🚨 제어 트리거
    # --------------------------------------------------------
    if invalid_count >= 3:
        print("⚠️ 기준 위반 3회 → 전류 보정 시작")

        target = nearest_even(angle)

        while angle != target:
            error = angle - target
            state = get_state(error)

            prev_current = current

            action = choose_action(state)
            current, delta_i = apply_action(current, action)

            # 전류 보정 → 각도 보정
            angle -= delta_i * ANGLE_PER_CURRENT

            print(
                f"  → 보정 중 | 목표각도: {target}°, "
                f"각도: {angle:.1f}°, "
                f"전류: {current:.2f}A (ΔI = {delta_i:+.2f}A)"
            )

            time.sleep(0.3)

        print("✅ 2의 배수 각도 복원 완료\n")
        invalid_count = 0

    time.sleep(1)



🚀 실시간 입력 시작

입력 각도: 10°, 전류: 1.00A
입력 각도: 6°, 전류: 0.60A
입력 각도: 5°, 전류: 0.50A
입력 각도: 10°, 전류: 1.00A
입력 각도: 7°, 전류: 0.70A
입력 각도: 3°, 전류: 0.30A
입력 각도: 6°, 전류: 0.60A
입력 각도: 1°, 전류: 0.10A


KeyboardInterrupt: 

In [ ]:
'''
각도 : 0~10 사이의 데이터가 랜덤하게 초당 1개씩 들어오고, 전류 : 각도에 0.1씩 곱한값으로 동일하게 초당 1개씩 입력되는 코드, 그리고 각도 정답 기준은 입력될 때마다 2씩 더한값이 출력이 되어야 해, 그래서 연속으로 3번 기준값인 2 더한값이 안나오고 다른값이 입력으로 들어오면 다음 값에서는 원래의 2더한값으로 맞춰야해, 그런데 맞추는 과정은 전류가 입력되는 값에 0.1씩 증가 또는 감소를 시도하면 각도도 1씩 증가, 감소가 되서 원래 각도 기준인 2더한값으로 맞출 수 있는 강화학습 모델 코드를 만들어줘
'''

In [14]:
import random
import time
from collections import defaultdict

# ============================================================
#  시스템 상수 정의
# ============================================================

ANGLE_MIN = 0
ANGLE_MAX = 10

# 전류 = 각도 × 0.1
CURRENT_GAIN = 0.1

# 전류 보정 step
CURRENT_STEP = 0.1

# 전류 0.1 → 각도 1 변화
ANGLE_PER_CURRENT = 10.0

# 행동 정의
# index 0: 전류 감소, 1: 유지, 2: 증가
ACTIONS = [-1, 0, 1]

# ============================================================
#  Q-Learning 설정
# ============================================================

# Q[state][action]
Q = defaultdict(lambda: [0.0, 0.0, 0.0])

alpha = 0.4      # 학습률
gamma = 0.6      # 할인율
epsilon = 0.2    # 탐험 확률

# ============================================================
#  유틸 함수
# ============================================================

def get_state(angle_error):
    """
    상태 정의
    - 현재 각도와 목표 기준 각도(target)의 차이
    """
    return int(angle_error)


def choose_action(state):
    """
    ε-greedy 정책
    """
    if random.random() < epsilon:
        return random.randint(0, 2)
    return max(range(3), key=lambda a: Q[state][a])


def apply_action(current, action_idx):
    """
    전류 보정 적용
    """
    delta_i = ACTIONS[action_idx] * CURRENT_STEP
    return current + delta_i, delta_i


# ============================================================
#  강화학습 사전 학습
# ============================================================

def train_agent(episodes=1200):
    """
    전류를 조정해서
    각도를 '기준 각도(target)'에 맞추는 학습
    """
    for _ in range(episodes):

        angle = random.randint(ANGLE_MIN, ANGLE_MAX)
        target = random.randint(0, 20)  # 기준 각도는 계속 증가 가능
        current = angle * CURRENT_GAIN

        error = angle - target

        while error != 0:
            state = get_state(error)
            action = choose_action(state)

            current, delta_i = apply_action(current, action)

            # 전류 변화 → 각도 변화
            angle -= delta_i * ANGLE_PER_CURRENT

            next_error = angle - target
            reward = -abs(next_error)
            next_state = get_state(next_error)

            Q[state][action] += alpha * (
                reward + gamma * max(Q[next_state]) - Q[state][action]
            )

            error = next_error


train_agent()

# ============================================================
#  실시간 데이터 스트림 + 제어
# ============================================================

invalid_count = 0
target_angle = 0  # 기준 각도 시작값

print("\n 실시간 입력 시작\n")

while True:
    # --------------------------------------------------------
    # 기준 각도 업데이트 (항상 +2)
    # --------------------------------------------------------
    target_angle += 2

    # --------------------------------------------------------
    # 입력 데이터 생성
    # --------------------------------------------------------
    angle = random.randint(ANGLE_MIN, ANGLE_MAX)
    current = angle * CURRENT_GAIN

    print(
        f"입력 각도: {angle}°, "
        f"전류: {current:.2f}A | "
        f"기준 각도: {target_angle}°"
    )

    # --------------------------------------------------------
    # 기준 위반 판단
    # --------------------------------------------------------
    if angle != target_angle:
        invalid_count += 1
    else:
        invalid_count = 0

    # --------------------------------------------------------
    #  강화학습 제어 트리거
    # --------------------------------------------------------
    if invalid_count >= 3:
        print(" 기준 각도 불일치 3회 → 전류 보정 시작")

        while angle != target_angle:
            error = angle - target_angle
            state = get_state(error)

            prev_current = current

            action = choose_action(state)
            current, delta_i = apply_action(current, action)

            # 전류 보정 → 각도 보정
            angle -= delta_i * ANGLE_PER_CURRENT

            print(
                f"  → 보정 중 | "
                f"목표각도: {target_angle}°, "
                f"각도: {angle:.1f}°, "
                f"전류: {current:.2f}A (ΔI = {delta_i:+.2f}A)"
            )

            time.sleep(0.3)

        print(" 기준 각도 복원 완료\n")
        invalid_count = 0

    time.sleep(1)



 실시간 입력 시작

입력 각도: 1°, 전류: 0.10A | 기준 각도: 2°
입력 각도: 8°, 전류: 0.80A | 기준 각도: 4°
입력 각도: 5°, 전류: 0.50A | 기준 각도: 6°
 기준 각도 불일치 3회 → 전류 보정 시작
  → 보정 중 | 목표각도: 6°, 각도: 6.0°, 전류: 0.40A (ΔI = -0.10A)
 기준 각도 복원 완료

입력 각도: 3°, 전류: 0.30A | 기준 각도: 8°
입력 각도: 7°, 전류: 0.70A | 기준 각도: 10°
입력 각도: 3°, 전류: 0.30A | 기준 각도: 12°
 기준 각도 불일치 3회 → 전류 보정 시작
  → 보정 중 | 목표각도: 12°, 각도: 3.0°, 전류: 0.30A (ΔI = +0.00A)
  → 보정 중 | 목표각도: 12°, 각도: 4.0°, 전류: 0.20A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 3.0°, 전류: 0.30A (ΔI = +0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 4.0°, 전류: 0.20A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 5.0°, 전류: 0.10A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 6.0°, 전류: 0.00A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 7.0°, 전류: -0.10A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 8.0°, 전류: -0.20A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 9.0°, 전류: -0.30A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 10.0°, 전류: -0.40A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 11.0°, 전류: -0.50A (ΔI = -0.10A)
  → 보정 중 | 목표각도: 12°, 각도: 10.0°, 전류: -0.40A (

KeyboardInterrupt: 